# GEPA for AIME Notebook

In this tutorial, we optimize GPT-4.1 Mini's Chain of Thought (dspy.ChainOfThought) for solving math problems (AIME) using the dspy.GEPA optimizer!


In [2]:
# Watsonx AI Credentials
import os
from ibm_watsonx_ai import APIClient,Credentials
api_key = os.getenv("WATSONX_API_KEY")
url = "https://us-south.ml.cloud.ibm.com"
project_id = os.getenv("WATSONX_PROJECT_ID")

#Testing Credentials 
credentials = Credentials(api_key=api_key, url=url)

client = APIClient(credentials)

client.set.default_project(project_id)

'SUCCESS'

In [3]:
# Flags

MLFlow_Active = False

In [4]:

if MLFlow_Active:
    # Increase connection pool settings before importing mlflow
    os.environ["MLFLOW_HTTP_POOL_CONNECTIONS"] = "50"  # number of pools
    os.environ["MLFLOW_HTTP_POOL_MAXSIZE"] = "50"  # maximum size of each pool

In [ ]:
# Setup DSPy

import dspy
model_name = "watsonx/openai/gpt-oss-120b"
max_tokens = 32000
temperature = 0.7

lm = dspy.LM(model=model_name, max_tokens=32000, api_key=api_key,tenperature = 0.7
             ,project_id = project_id
             ,api_base=url
             )

dspy.configure(lm=lm, temperature=1, trace=[], experimental=True)

In [6]:
# MLFlow Integration with DSPy 


if MLFlow_Active:
    # Setup MLFlow
    import mlflow


    # To Run the Server 
    # mlflow server --backend-store-uri sqlite:///mydb.sqlite

    # Enable autologging with all features
    mlflow.dspy.autolog(
        log_compiles=True,    # Track optimization process
        log_evals=True,       # Track evaluation results
        log_traces_from_compile=True  # Track program traces during optimization
    )

    # Configure MLflow tracking
    mlflow.set_tracking_uri("http://localhost:5000")  # Use local MLflow server
    mlflow.set_experiment("GEPA_AIME_Optimization (BASE)")  # Set experiment name

### Loading the AIME dataset

The AIME exam consists of 2 problem sets of size 15 for each year. For this tutorial, we will use AIME problem sets from previous years (2022-2024) for optimization (amounting to total 3 years x 2 sets x 15 problems = 90 problems, split equally between train and validation sets), and test the performance on AIME 2025 (2 sets x 15 problems = 30 problems). Since AIME 2025 is a small set, we repeat it 5 times for statistical stability in evaluation.

In [7]:
import dspy
from datasets import load_dataset

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            'solution': x['solution'],
            'answer': x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    import random
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            'answer': x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[:int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

c:\workspace\AssetOpsBench\notebook\.venv_gepa\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

(45, 45, 150)

Let's view an example task input

In [9]:
print("Problem:")
print(train_set[0]['problem'])
print("\n\nSolution:")
print(train_set[0]['solution'])
print("\n\nAnswer:")
print(train_set[0]['answer'])

Problem:
In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.


Solution:
We have the following diagram:

Let $X$ and $W$ be the points where $AP$ and $BQ$ extend to meet $CD$, and $YZ$ be the height of $\triangle AZB$. As proven in Solution 2, triangles $APD$ and $DPW$ are congruent right triangles. Therefore, $AD = DW = 333$. We can apply this logic to triangles $BCQ$ and $XCQ$ as well, giving us $BC = CX = 333$. Since $CD = 650$, $XW = DW + CX - CD = 16$.
Additionally, we can see that $\triangle XZW$ is similar to $\triangle PQZ$ and $\triangle AZB$. We know that $\frac{XW}{AB} = \frac{16}{500}$. So, we can say that the height of the triangle $AZB$ is $500u$ while the height of the triangle $XZW$ is $16u$. After that, we can figure out the distance from 

Let's define the program: A simple `dspy.ChainOfThought`

In [10]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

### Defining the evaluation metric
We simply check exact match between the predicted answer and the correct answer.

In [11]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError as e:
        return 0
    return int(correct_answer == llm_answer)

Evaluating unoptimized Chain Of Thought

In [ ]:
import dspy
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True
)

evaluate(program)

Average Metric: 1.00 / 1 (100.0%):   1%|          | 1/150 [00:08<21:12,  8.54s/it]

### Optimize the program with `dspy.GEPA`

GEPA is a reflective prompt optimizer, and it's strength lies in being able to leverage additional sources of information, like the DSPy program's execution and evaluation pipelines, which provides GEPA more visibility into why the system got the score that it did, and then GEPA can introspect to identify how to improve the score. GEPA can also leverage additional supervision provided in this manner. For example, during optimization, we can return the correct solution's to the problems the program failed to solve.

We note that while such explicit supervision is not available in all scenarios, GEPA can work very flexibly with different forms of feedback (for example, using LLM-as-a-judge feedback shown in the PAPILLON tutorial, or just using answer labels, as shown in the facility-support tutorial).

Let's quickly modify the evaluation metric to become an optimization metric for GEPA, that can provide this additional supervision!

In [ ]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    try:
        llm_answer = int(prediction.answer)
    except ValueError as e:
        feedback_text = f"The final answer must be a valid integer and nothing else. You responded with '{prediction.answer}', which couldn't be parsed as a python integer. Please ensure your answer is a valid integer without any additional text or formatting."
        feedback_text += f" The correct answer is '{correct_answer}'."
        if written_solution:
            feedback_text += f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways you can learn from this solution to improve your future answers and approach to similar problems and ensure your final answer is a valid integer."
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)

    print(score)
    
    feedback_text = ""
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{correct_answer}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{correct_answer}'."
    
    if written_solution:
        feedback_text += f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways you can learn from this solution to improve your future answers and approach to similar problems."

    return dspy.Prediction(score=score, feedback=feedback_text)

In [ ]:
from dspy import GEPA


if MLFlow_Active:
    mlflow.set_experiment("GEPA_AIME_Optimization (Experiment)")  # Set experiment name


optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=32,
    track_stats=True,
    reflection_minibatch_size=3,
    reflection_lm=dspy.LM(model=model_name, max_tokens=32000, api_key=api_key,tenperature = 0.7
             ,project_id = project_id
             ,api_base=url)
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2026/02/28 19:28:35 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '644fcc86ba314414b254a901d03cfad2', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current dspy workflow
2026/02/28 19:28:35 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 560 metric calls of the program. This amounts to 6.22 full evals on the train+val set.
2026/02/28 19:28:35 INFO dspy.teleprompt.gepa.gepa: Using 45 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/560 [00:00<?, ?rollouts/s]

1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
0
1
1
1
1
0
1
1
1
1
1
1
0
0
1
1
1
1
0
1
1
1
1
1
1


2026/02/28 19:31:42 INFO dspy.evaluate.evaluate: Average Metric: 37.0 / 45 (82.2%)


0


2026/02/28 19:31:43 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.8222222222222222 over 45 / 45 examples
GEPA Optimization:   8%|▊         | 45/560 [03:07<35:50,  4.18s/rollouts]2026/02/28 19:31:43 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.8222222222222222


...

1
1
1
1
1
1
1
1
1
1
1
1
1
1


2026/02/28 19:40:02 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'


1
1
0
1
0
1
0
1
0
1
1
1
1
1
0
1
0


2026/02/28 19:41:53 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'Let $a, b, x,$ and $y$ be real numbers with $a>4$ and $b>1$ such that\\[\\frac{x^2}{a^2}+\\frac{y^2}{a^2-16}=\\frac{(x-20)^2}{b^2-1}+\\frac{(y-11)^2}{b^2}=1.\\]Find the least possible value of $a+b.$', 'solution': 'Denote $P = \\left( x , y \\right)$.\nBecause $\\frac{x^2}{a^2}+\\frac{y^2}{a^2-16} = 1$, $P$  is on an ellipse whose center is $\\left( 0 , 0 \\right)$ and foci are $\\left( - 4 , 0 \\right)$ and $\\left( 4 , 0 \\right)$.\nHence, the sum of distance from $P$ to $\\left( - 4 , 0 \\right)$ and $\\left( 4 , 0 \\right)$ is equal to twice the major axis of this ellipse, $2a$.\nBecause $\\frac{(x-20)^2}{b^2-1}+\\frac{(y-11)^2}{b^2} = 1$, $P$  is on an ellipse whose center is $\\left( 20 , 11 \\right)$ and foci are $\\left( 20 , 10 \\right)$ and $\\left( 20 , 12 \\right)$.\nHence, the sum of distance from $P$ to $\\left( 20 , 10 \\right)$ and $\\left( 20 , 12 \\right)$ is equal to twice the major axi

1


Let's see the prompt generated

In [ ]:
print(optimized_program.predict.signature.instructions)

It can be seen that what GEPA is doing here, is precomputing some reasoning to come up with a good plan for future task instances. Due to the improved performance in unseen validation set, we expect this prompt to generalize!

Evaluating the Chain Of Thought optimized with GEPA

In [ ]:
evaluate(optimized_program)